In [8]:
# Bibliotecas padrão
import os
import gc
import json
import shutil
import zipfile

# Bibliotecas de terceiros
import ee
import geemap
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, silhouette_samples
from sklearn.mixture import GaussianMixture
from shapely.geometry import box
from sklearn.cluster import KMeans
from io import BytesIO
from glob import glob

print("Bibliotecas carregadas")

Bibliotecas carregadas


In [10]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
ee.Authenticate()
ee.Initialize(project="spatial-yew-490017-r3")
print(ee.String("Hello from the Earth Engine servers!").getInfo())

Hello from the Earth Engine servers!


In [4]:
FILES_DIR = "files"
os.makedirs(FILES_DIR, exist_ok=True)

PRINTS_DIR = os.path.join(FILES_DIR, "prints")
os.makedirs(PRINTS_DIR, exist_ok=True)

In [35]:
METRIC_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic

In [39]:
# Limites das UCs federais, via serviço WFS da INDE/ICMBio

ucs_dir = os.path.join(FILES_DIR, "limites_ucs")
os.makedirs(ucs_dir, exist_ok=True)

uc_typename = "ICMBio:limiteucsfederais_a"
uc_output_path = os.path.join(ucs_dir, "limites_ucs.geojson")

if os.path.exists(uc_output_path):
    print(f"Already downloaded: {uc_output_path}")
else:
    wfs_url = (
        "https://geoservicos.inde.gov.br/geoserver/ICMBio/ows"
        f"?service=WFS&version=2.0.0&request=GetFeature"
        f"&typeName={uc_typename}&outputFormat=application/json"
    )
    response = requests.get(wfs_url, timeout=180, verify=False)
    response.raise_for_status()
    with open(uc_output_path, "wb") as f:
        f.write(response.content)
    print(f"Saved: {uc_output_path}")

gdf_ucs = gpd.read_file(uc_output_path)

# Reprojetar para CRS métrico
gdf_ucs = gdf_ucs.to_crs("EPSG:5880")
gdf_ucs["geometry"] = gdf_ucs.geometry.buffer(0)  # corrige geometrias inválidas

Already downloaded: files\limites_ucs\limites_ucs.geojson


In [41]:

print("Shape:", gdf_ucs.shape)
print("Columns:", gdf_ucs.columns.tolist())
print("Descriptions:", gdf_ucs.describe())
print("Descriptions:", gdf_ucs.info())
print("CRS:", gdf_ucs.crs)

Shape: (347, 22)
Columns: ['ogc_fid', 'id', 'nomeuc', 'cnuc', 'criacaoano', 'areahaalb', 'perimm', 'criacaoato', 'esferaadm', 'grupouc', 'biomas', 'gregional', 'fusoabrang', 'demarcacao', 'escalauc', 'bioma_pred', 'cat_iucn', 'uf', 'categoria_', 'sigla_cate', 'dominio', 'geometry']
Descriptions:           ogc_fid           id     areahaalb        perimm
count  347.000000   347.000000  3.470000e+02  3.470000e+02
mean   174.000000  1919.069164  4.998011e+05  3.585859e+05
std    100.314505   104.076566  3.033114e+06  5.263453e+05
min      1.000000  1739.000000  6.517440e+01  4.417637e+03
25%     87.500000  1827.500000  8.688228e+03  6.932556e+04
50%    174.000000  1921.000000  5.583451e+04  2.053673e+05
75%    260.500000  2008.500000  2.852276e+05  4.235245e+05
max    347.000000  2099.000000  4.038442e+07  4.021336e+06
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 22 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      

# AAF (ICMBio)

### Importação

In [2]:
# AAF — download via ArcGIS FeatureServer (anos 2010-2026)

BASE_URL = "https://services3.arcgis.com/KYEMegXJrTiWSYWk/arcgis/rest/services"
PAGE_SIZE = 2000

aaf_urls_by_year = {}

# Padrão dinâmico — anos 2010 a 2022 seguem a mesma convenção de nome
for year in range(2010, 2023):
    aaf_urls_by_year[year] = (
        f"{BASE_URL}/db_geo_compartilhado_dmif_fogo_aaf_{year}/FeatureServer/0"
    )

# Exceções — nomes de serviço mudaram ano a ano a partir de 2023
aaf_urls_by_year[2023] = f"{BASE_URL}/AAF_2023_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2024] = f"{BASE_URL}/AAF_2024_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2025] = f"{BASE_URL}/AAF_2025_DGEO_oficial/FeatureServer/0"
aaf_urls_by_year[2026] = f"{BASE_URL}/AAF_2026/FeatureServer/0"

print("Total de anos mapeados:", len(aaf_urls_by_year))

Total de anos mapeados: 17


In [5]:
# Download com paginação, salvando um GeoJSON por ano

aaf_dir = os.path.join(FILES_DIR, "aaf")
os.makedirs(aaf_dir, exist_ok=True)

aaf_downloaded_files = {}

for year, base_url in aaf_urls_by_year.items():

    output_path = os.path.join(aaf_dir, f"aaf_{year}.geojson")

    if os.path.exists(output_path):
        aaf_downloaded_files[year] = output_path
        continue

    # Paginação — segue baixando enquanto o servidor retornar página cheia
    all_features = []
    offset = 0

    while True:
        query_url = (
            f"{base_url}/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
            f"&resultOffset={offset}&resultRecordCount={PAGE_SIZE}"
        )
        response = requests.get(query_url, timeout=120)
        response.raise_for_status()
        page_data = response.json()

        features = page_data.get("features", [])
        if not features:
            break

        all_features.extend(features)

        if len(features) < PAGE_SIZE:
            break

        offset += PAGE_SIZE

    # Salvar como GeoJSON válido
    final_geojson = {"type": "FeatureCollection", "features": all_features}

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_geojson, f)

    aaf_downloaded_files[year] = output_path
    print(f"[{year}] Salvo — {len(all_features)} registros")

print("\nAnos baixados:", list(aaf_downloaded_files.keys()))


Anos baixados: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [56]:
# Carregar todos os anos, checando consistência de colunas antes de concatenar

aaf_gdf_list = []
reference_columns = None

for year, path in aaf_downloaded_files.items():
    gdf_year = gpd.read_file(path)

    if reference_columns is None:
        reference_columns = set(gdf_year.columns)
    else:
        diff = set(gdf_year.columns).symmetric_difference(reference_columns)
        if diff:
            print(f"[{year}] Diferença de colunas em relação ao primeiro ano: {diff}")

    gdf_year["file_year"] = year
    aaf_gdf_list.append(gdf_year)

gdf_aaf = gpd.GeoDataFrame(pd.concat(aaf_gdf_list, ignore_index=True))


print("Shape:", gdf_aaf.shape)
print("Colunas:", gdf_aaf.columns.tolist())
print("Anos presentes:", sorted(gdf_aaf["file_year"].unique()))

[2024] Diferença de colunas em relação ao primeiro ano: {'ano', 'data_img', 'ngi', 'gr_nome', 'cr', 'area_uc', 'data', 'local', 'area_ent', 'pk_aaf', 'bioma', 'FID', 'mes_num', 'ct', 'mes_nome', 'categoria', 'GlobalID'}
[2025] Diferença de colunas em relação ao primeiro ano: {'ano', 'data_img', 'ngi', 'gr_nome', 'cr', 'area_uc', 'data', 'local', 'area_ent', 'acao', 'pk_aaf', 'bioma', 'FID', 'Shape__Are', 'mes_num', 'ct', 'tipo', 'mes_nome', 'categoria', 'GlobalID', 'Shape__Len', 'classe'}
[2026] Diferença de colunas em relação ao primeiro ano: {'ano', 'data_img', 'ngi', 'gr_nome', 'cr', 'area_uc', 'data', 'local', 'area_ent', 'acao', 'pk_aaf', 'bioma', 'FID', 'Shape__Are', 'GlobalID_2', 'mes_num', 'ct', 'tipo', 'mes_nome', 'categoria', 'GlobalID', 'Shape__Len', 'classe'}
Shape: (41780, 33)
Colunas: ['pk_aaf', 'cnuc', 'nome_uc', 'area_ha', 'data_img', 'classe', 'satelite', 'obs', 'juliano', 'Shape__Area', 'Shape__Length', 'geometry', 'file_year', 'FID', 'data', 'ano', 'mes_nome', 'mes_n

### Tratamento dos dados

In [ ]:
# Reconstrução de data com granularidade decrescente, sem descarte de registros
#
# Descoberta: 2010 inteiro (354 registros) e parte de 2021 (32 registros) não têm
# nenhuma fonte de data, nem dia juliano. 'juliano' também tem 918 valores sentinela
# fora do intervalo 1-366 (incluindo 0 e 2016) que precisam ser filtrados antes de
# virar fallback, senão geram event_date fantasiosa. Nenhum evento é descartado por
# falta de data, event_year sempre existe via file_year, event_date fica NaT quando
# não há nenhuma fonte válida.

date_from_new_schema = pd.to_datetime(gdf_aaf["data"], unit="ms", errors="coerce")
date_from_old_schema = pd.to_datetime(gdf_aaf["data_img"], errors="coerce")

julian_day = pd.to_numeric(gdf_aaf["juliano"], errors="coerce")
julian_day = julian_day.where(julian_day.between(1, 366))
date_from_julian = (
    pd.to_datetime(gdf_aaf["file_year"].astype(str), format="%Y", errors="coerce")
    + pd.to_timedelta(julian_day - 1, unit="D")
)

gdf_aaf["event_date"] = (
    date_from_new_schema
    .combine_first(date_from_old_schema)
    .combine_first(date_from_julian)
)

gdf_aaf["event_year"] = gdf_aaf["file_year"]
gdf_aaf["event_month"] = gdf_aaf["event_date"].dt.month

gdf_aaf["date_precision"] = np.select(
    condlist=[
        date_from_new_schema.notna(),
        date_from_old_schema.notna(),
        date_from_julian.notna(),
    ],
    choicelist=["day_epoch", "day_img", "day_julian"],
    default="year_only",
)

print(gdf_aaf["date_precision"].value_counts())



date_precision
day_img       26954
day_epoch     12724
day_julian     1517
year_only       585
Name: count, dtype: int64


In [ ]:
# Checagem pós-reconstrução de data

print("=== Distribuição de date_precision por ano ===")
precision_por_ano = pd.crosstab(gdf_aaf["file_year"], gdf_aaf["date_precision"])
print(precision_por_ano)

print("\n=== event_date ainda nula (esperado: só quem não tinha nenhuma fonte) ===")
sem_data = gdf_aaf[gdf_aaf["event_date"].isna()]
print(f"Total: {len(sem_data)}")
print(sem_data["file_year"].value_counts().sort_index())

print("\n=== Consistência: event_year bate com o ano de event_date, quando existe ===")
tem_data_completa = gdf_aaf["event_date"].notna()
ano_da_data = gdf_aaf.loc[tem_data_completa, "event_date"].dt.year
divergente = gdf_aaf.loc[tem_data_completa, "event_year"] != ano_da_data
print(f"Casos divergentes: {divergente.sum()} de {tem_data_completa.sum()}")
if divergente.sum() > 0:
    print(gdf_aaf.loc[tem_data_completa][divergente][["file_year", "event_date", "event_year", "date_precision"]].head(10))

print("\n=== event_month, distribuição geral (checar sazonalidade plausível) ===")
print(gdf_aaf["event_month"].value_counts(dropna=False).sort_index())

print("\n=== Faixa de datas resultante por precisão ===")
for precisao in gdf_aaf["date_precision"].unique():
    if precisao == "year_only":
        continue
    subset = gdf_aaf.loc[gdf_aaf["date_precision"] == precisao, "event_date"]
    print(f"{precisao}: {subset.min()} até {subset.max()}")

=== Distribuição de date_precision por ano ===
date_precision  day_epoch  day_img  day_julian  year_only
file_year                                                
2010                    0        0           0        354
2011                    0       65        1517          1
2012                    0     1628           0          0
2013                    0      953           0          0
2014                    0     1397           0          0
2015                    0     2088           0          0
2016                    0     2025           0          0
2017                    0     2692           0          0
2018                    0     1316           0        197
2019                    0     2162           0          0
2020                    0     2486           0          1
2021                    0     2130           0         32
2022                    0     2768           0          0
2023                    0     5244           0          0
2024                 7503

In [49]:
#  Reprojeção e correção de geometria

gdf_aaf = gdf_aaf.to_crs(METRIC_CRS)
gdf_aaf["geometry"] = gdf_aaf.geometry.buffer(0)

n_invalid = (~gdf_aaf.is_valid).sum()
print(f"Geometrias inválidas após buffer(0): {n_invalid}")
gdf_aaf = gdf_aaf[gdf_aaf.is_valid].copy()

Geometrias inválidas após buffer(0): 2


In [50]:
# Área e perímetro recalculados direto da geometria
# Shape__Area original oscila entre grau² e m² sem padrão fixo por ano
# (confirmado em 2024/2025 vs o resto), não é confiável em nenhum caso

gdf_aaf["area_ha_original"] = gdf_aaf["area_ha"]
gdf_aaf["area_ha"] = gdf_aaf.geometry.area / 10_000
gdf_aaf["shape_length"] = gdf_aaf.geometry.length

In [ ]:
# Limpeza de string "None" tratada como categoria válida

categorical_columns = ["classe", "acao", "tipo", "categoria"]
for column in categorical_columns:
    gdf_aaf[column] = gdf_aaf[column].replace("None", np.nan)

In [51]:
# Harmonização do tipo de evento
# classe cobre 2020-2024, acao cobre 2025-2026, nenhuma cobre 2010-2019
# tipo e categoria são dimensões diferentes, não entram nessa fusão

harmonization_map = {
    "aceiro": "aceiro",
    "fogo natural": "natural",
    "raio": "natural",
    "gestao de ignicao natural": "natural",
    "incendio": "incendio",
    "indigena": "antropica",
    "gestao de ignicao antropica": "antropica",
    "queima por indigenas isolados": "antropica",
    "outros": "outros",
    "queima controlada": "queima controlada",
    "queima prescrita": "queima prescrita",
}

gdf_aaf["event_type"] = (
    gdf_aaf["classe"]
    .combine_first(gdf_aaf["acao"])
    .map(harmonization_map)
)
gdf_aaf["management_type"] = gdf_aaf["tipo"]
gdf_aaf["response_category"] = gdf_aaf["categoria"]

In [52]:
# Remoção de duplicata geométrica exata

n_before = len(gdf_aaf)
gdf_aaf = gdf_aaf[
    ~gdf_aaf.geometry.apply(lambda geom: geom.wkb).duplicated()
].copy()
print(f"Duplicatas geométricas removidas: {n_before - len(gdf_aaf)}")

Duplicatas geométricas removidas: 283


In [54]:
# Backfill administrativo via overlay espacial com as UCs
# cnuc, bioma e area_uc são a mesma grandeza nas duas fontes, entram por combine_first

gdf_aaf = gdf_aaf.reset_index(drop=True)
gdf_aaf["event_id"] = gdf_aaf.index

admin_columns = ["cnuc", "bioma", "area_uc"]
needs_backfill = gdf_aaf[admin_columns].isna().any(axis=1)

events_to_backfill = gdf_aaf.loc[needs_backfill, ["event_id", "geometry"]]
uc_columns = ["cnuc", "bioma_pred", "areahaalb", "categoria_"]

overlay_result = gpd.overlay(
    events_to_backfill, gdf_ucs[uc_columns + ["geometry"]], how="intersection"
)
overlay_result["intersection_area"] = overlay_result.geometry.area

best_match = (
    overlay_result
    .sort_values("intersection_area", ascending=False)
    .drop_duplicates(subset="event_id", keep="first")
    .set_index("event_id")
)

gdf_aaf["cnuc"] = gdf_aaf["cnuc"].combine_first(gdf_aaf["event_id"].map(best_match["cnuc"]))
gdf_aaf["bioma"] = gdf_aaf["bioma"].combine_first(gdf_aaf["event_id"].map(best_match["bioma_pred"]))
gdf_aaf["area_uc"] = gdf_aaf["area_uc"].combine_first(gdf_aaf["event_id"].map(best_match["areahaalb"]))


print(f"Eventos com backfill administrativo via overlay: {best_match['cnuc'].notna().sum()} de {needs_backfill.sum()}")

Eventos com backfill administrativo via overlay: 27357 de 28895


c:\Users\ANDERSONALVESCOELHO\miniconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:375: UserWarning: `keep_geom_type=True` in overlay resulted in 9 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [ ]:
# Limpeza final de colunas

dead_columns = ["Shape__Are", "Shape__Len", "GlobalID_2", "Shape__Area", "Shape__Length", "event_id"]
gdf_aaf = gdf_aaf.drop(columns=[c for c in dead_columns if c in gdf_aaf.columns])

print("Shape final:", gdf_aaf.shape)
print("\nDistribuição de event_type:")
print(gdf_aaf["event_type"].value_counts(dropna=False))

Shape final: (41495, 33)

Distribuição de event_type:
event_type
incendio             17076
NaN                  16187
queima prescrita      5743
queima controlada     1697
antropica              371
natural                212
aceiro                 105
outros                 104
Name: count, dtype: int64


# MapBiomas Fogo Collection 4 

In [11]:
mapbiomas_fire_annual = ee.Image(
    "projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_annual_burned_v1"
)

mapbiomas_fire_scar_size = ee.Image(
    "projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_annual_burned_scar_size_range_v1"
)

print("Bandas (área queimada anual):", mapbiomas_fire_annual.bandNames().getInfo())
print("Bandas (tamanho de cicatriz e frequência):", mapbiomas_fire_scar_size.bandNames().getInfo())

Bandas (área queimada anual): ['burned_area_1985', 'burned_area_1986', 'burned_area_1987', 'burned_area_1988', 'burned_area_1989', 'burned_area_1990', 'burned_area_1991', 'burned_area_1992', 'burned_area_1993', 'burned_area_1994', 'burned_area_1995', 'burned_area_1996', 'burned_area_1997', 'burned_area_1998', 'burned_area_1999', 'burned_area_2000', 'burned_area_2001', 'burned_area_2002', 'burned_area_2003', 'burned_area_2004', 'burned_area_2005', 'burned_area_2006', 'burned_area_2007', 'burned_area_2008', 'burned_area_2009', 'burned_area_2010', 'burned_area_2011', 'burned_area_2012', 'burned_area_2013', 'burned_area_2014', 'burned_area_2015', 'burned_area_2016', 'burned_area_2017', 'burned_area_2018', 'burned_area_2019', 'burned_area_2020', 'burned_area_2021', 'burned_area_2022', 'burned_area_2023', 'burned_area_2024']
Bandas (tamanho de cicatriz e frequência): ['scar_area_ha_1985', 'scar_area_ha_1986', 'scar_area_ha_1987', 'scar_area_ha_1988', 'scar_area_ha_1989', 'scar_area_ha_1990',

# MODIS MCD64A1 

In [ ]:
# MODIS MCD64A1 — leitura prévia via GEE

modis_burned_area = ee.ImageCollection("MODIS/061/MCD64A1").filterDate("2003-01-01", "2025-12-31")

print("Total de imagens:", modis_burned_area.size().getInfo())
print("Bandas:", modis_burned_area.first().bandNames().getInfo())

Total de imagens: 276
Bandas: ['BurnDate', 'Uncertainty', 'QA', 'FirstDay', 'LastDay']


# FireCCI51 (ESA)

In [13]:
# Série parada em 2020-12, não alcança o período recente do AAF, útil só como referência retrospectiva

firecci = ee.ImageCollection("ESA/CCI/FireCCI/5_1")

print("Total de imagens:", firecci.size().getInfo())
print("Bandas:", firecci.first().bandNames().getInfo())

Total de imagens: 240
Bandas: ['BurnDate', 'ConfidenceLevel', 'LandCover', 'ObservedFlag']


# GABAM 

In [15]:
gabam = ee.ImageCollection("projects/sat-io/open-datasets/GABAM")

print("Total de imagens:", gabam.size().getInfo())
print("Bandas:", gabam.first().bandNames().getInfo())

Total de imagens: 14614
Bandas: ['b1']


In [ ]:
# LASA-Alarmes (UFRJ) — sem asset GEE confirmado e sem API pública documentada
# Acesso é via plataforma própria (alarmes.lasa.ufrj.br), possivelmente exige contato direto com o LASA
# Cobertura hoje é regional, V1 beta pra Cerrado/Pantanal e protótipo pra Roraima, não é nacional
# Deixando como placeholder até confirmar via contato com o laboratório

lasa_alarmes_status = "acesso não confirmado — requer contato direto com LASA/UFRJ"

print(lasa_alarmes_status)